# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields using @id for reference
print("Available Record Sets (by @id):")
record_sets = [r for r in metadata.record_sets]
for rs in record_sets:
    print(f"  - {rs['@id']} (name: {rs.get('name', 'N/A')})")
    if 'fields' in rs:
        print("    Fields:")
        for field in rs['fields']:
            print(f"      - {field['@id']} (name: {field.get('name', 'N/A')}, type: {field.get('dataType', 'N/A')})")
print()
# View a few records from a sample record set (by @id)
if len(record_sets) > 0:
    print("Sample record from the first record set:")
    example_rs_id = record_sets[0]['@id']
    for i, record in enumerate(dataset.records(record_set=example_rs_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from all record sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Display available columns for the first record set
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id and main_rs_id in dataframes:
    print(f"Columns in RecordSet {main_rs_id}:\n{dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select the main DataFrame and one of its numeric fields for analysis
if main_rs_id and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    # Try to auto-detect a numeric field
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_candidates:
        # Try to coerce numeric fields from those named 'coverage', 'peptide_count', 'MW', etc.
        for pattern in ['coverage', 'peptide', 'MW', 'molecular_weight', 'abundance', 'count']:
            candidates = [c for c in df.columns if pattern in c.lower()]
            for c in candidates:
                df[c] = pd.to_numeric(df[c], errors='coerce')
        numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Pick the first one
        print(f"Using numeric field: {numeric_field}")
        threshold = np.nanmean(df[numeric_field])  # Use mean for threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (Mean as threshold): {len(filtered_df)} records found")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by a likely categorical field if present
        group_candidates = [c for c in df.columns if c not in numeric_candidates and pd.api.types.is_object_dtype(df[c])]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric fields found for analysis.")
else:
    print("No main DataFrame available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize the numeric field distribution and group means if possible
if main_rs_id and main_rs_id in dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8,5))
    df[numeric_field].hist(bins=30)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,6))
        grouped_df.plot(x=group_field, y=numeric_field, kind='bar', legend=False, ax=plt.gca())
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains record sets and fields on protein attributes from mass spectrometry analysis of extracellular vesicles.
- We demonstrated how to load data using the `mlcroissant` interface and dynamically explore record sets and fields using their `@id`.
- Simple EDA showed filtering and normalizing a numeric field, and visualizing distributions. For in-depth analysis, consult the dataset documentation and adapt the workflow to your needs.